# Практическая работа 4
## Введение в RAG и базовый пайплайн

### Архитектура RAG

```
Документы --> Chunking --> Embeddings --> Индекс (FAISS)
                                               |
Вопрос ----> Embedding --> Retrieval (top-k) --+
                                               |
                               Augmented Prompt
                                               |
                                     LLM Generation
                                               |
                                            Ответ
```

Перед запуском надо скачать модель:

```bash
ollama pull qwen2.5:1.5b
```

Если сервер Ollama уже запущен, этого достаточно.

| Компонент | Описание |
|---|---|
| Документы | исходный корпус текстов |
| Chunking | разбиение на фрагменты фиксированной длины с перекрытием |
| Embeddings | векторное представление каждого чанка |
| Индекс | структура для быстрого поиска ближайших соседей |
| Retrieval | поиск top-k наиболее близких чанков по запросу |
| Augmentation | формирование промпта с найденным контекстом |
| Generation | генерация ответа LLM строго на основе контекста |

---
## 1. Подготовка окружения

In [56]:
# ================================================================
# ЗАВИСИМОСТИ
#
# Базовый режим (автономный, без интернета):
#   TF-IDF эмбеддинги (sklearn) + FAISS индекс
#
# Для нейросетевых эмбеддингов в Colab раскомментируйте:
#   !pip install sentence-transformers faiss-cpu -q
#   USE_SENTENCE_TRANSFORMERS = True
#
# Для локальной генерации через Ollama:
#   !pip install ollama -q
#   ollama pull qwen2.5:1.5b
# ================================================================

import subprocess, sys

for pkg, module in [('faiss-cpu', 'faiss'), ('scikit-learn', 'sklearn'),
                    ('numpy', 'numpy'), ('matplotlib', 'matplotlib'),
                    ('ollama', 'ollama')]:
    try:
        __import__(module)
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])

import os, json, re, time, textwrap
import numpy as np
import faiss
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# --- Конфигурация ---
USE_SENTENCE_TRANSFORMERS = False   # True если доступен интернет и установлен пакет
EMB_MODEL_NAME            = 'all-MiniLM-L6-v2'

# Параметры chunking
CHUNK_SIZE    = 600   # символов
CHUNK_OVERLAP = 120   # символов перекрытия
TOP_K         = 3     # число чанков для retrieval

DOCS_DIR = 'docs'
os.makedirs(DOCS_DIR, exist_ok=True)

if USE_SENTENCE_TRANSFORMERS:
    from sentence_transformers import SentenceTransformer
    print(f'Режим эмбеддингов: sentence-transformers ({EMB_MODEL_NAME})')
else:
    print('Режим эмбеддингов: TF-IDF (автономный)')

import ollama
print('LLM: Ollama (qwen2.5:1.5b)')

print('Окружение готово.')

Режим эмбеддингов: TF-IDF (автономный)
LLM: Ollama (qwen2.5:1.5b)
Окружение готово.


---
## 2. Загрузка документов
### 2.1 Корпус из 7 документов по теме инжиниринга данных

In [57]:
CORPUS = [
    {
        'id': 'doc_01', 'title': 'Форматы данных: CSV, JSON, XML',
        'text': (
            "CSV (Comma-Separated Values) - текстовый формат хранения табличных данных. "
            "Каждая строка соответствует одной записи, поля разделяются запятыми. "
            "Преимущества CSV: простота, универсальность, читаемость человеком. "
            "Недостатки CSV: большой размер файла, отсутствие типов данных, отсутствие схемы. "
            "CSV применяется для экспорта данных и небольших датасетов.\n\n"
            "JSON (JavaScript Object Notation) - текстовый формат для структурированных данных. "
            "Поддерживает вложенность, массивы, объекты. Широко используется в веб-API. "
            "Преимущества JSON: гибкость, поддержка иерархии. "
            "Недостатки: избыточность, медленная обработка больших объемов.\n\n"
            "XML (eXtensible Markup Language) - формат с тегами и атрибутами. "
            "Поддерживает схемы валидации (XSD) и иерархическую структуру. "
            "Применяется в корпоративных системах и обмене данными между системами. "
            "Недостатки XML: многословность и сложность парсинга. "
            "При выборе формата нужно учитывать объем данных, структуру и требования к производительности."
        )
    },
    {
        'id': 'doc_02', 'title': 'Apache Parquet: колоночный формат',
        'text': (
            "Apache Parquet - бинарный колоночно-ориентированный формат хранения данных. "
            "Разработан для эффективного хранения и аналитической обработки больших данных.\n\n"
            "Ключевое свойство Parquet: данные каждой колонки хранятся непрерывно на диске. "
            "При аналитических запросах читаются только нужные колонки, а не вся строка. "
            "Это ускоряет операции AVG, SUM, GROUP BY в десятки раз по сравнению с CSV.\n\n"
            "Parquet поддерживает сжатие: snappy (быстрое), gzip (компактное), zstd. "
            "Формат хранит метаданные схемы и min/max статистики для каждого row-group. "
            "Это позволяет делать predicate pushdown - пропускать блоки данных без чтения.\n\n"
            "Parquet интегрируется со Spark, Hive, Presto, BigQuery, Athena, Delta Lake. "
            "Используется для Data Lake, DWH и аналитических пайплайнов. "
            "Типичная экономия места составляет от 3 до 8 раз по сравнению с CSV."
        )
    },
    {
        'id': 'doc_03', 'title': 'Apache Avro: формат сериализации',
        'text': (
            "Apache Avro - бинарный формат сериализации данных с поддержкой эволюции схем. "
            "Схема хранится прямо внутри файла в формате JSON.\n\n"
            "Главное преимущество Avro - поддержка эволюции схем. "
            "Backward-совместимость: новая схема может читать данные, записанные по старой схеме. "
            "Forward-совместимость: старая схема может читать данные по новой схеме. "
            "Для этого новые поля объявляются с default-значениями.\n\n"
            "Avro использует строчную модель хранения (row-based), что делает его "
            "оптимальным для записи потоков событий в Kafka и Flume. "
            "В Kafka-экосистеме Avro применяется вместе со Schema Registry.\n\n"
            "Типичный паттерн в Data Engineering: ingestion через Kafka с Avro-сериализацией, "
            "затем долгосрочное хранение в Parquet для аналитики."
        )
    },
    {
        'id': 'doc_04', 'title': 'Apache Spark: основы обработки данных',
        'text': (
            "Apache Spark - система распределённой обработки данных в памяти. "
            "Spark обеспечивает ускорение до 100x по сравнению с Hadoop MapReduce.\n\n"
            "Основные абстракции Spark: "
            "RDD (Resilient Distributed Dataset) - базовая неизменяемая распределённая коллекция. "
            "DataFrame - структурированный API поверх RDD, оптимизированный через Catalyst. "
            "Dataset - типобезопасный API (только Scala и Java).\n\n"
            "Операции Spark: трансформации (ленивые) - map, filter, groupBy, join; "
            "действия (запускают выполнение) - collect, count, show, write. "
            "Spark SQL позволяет выполнять SQL-запросы поверх DataFrame. "
            "DataFrame API и Spark SQL используют один Catalyst-оптимизатор, "
            "поэтому их производительность идентична.\n\n"
            "Spark интегрируется с HDFS, S3, Parquet, Avro, Kafka, Hive."
        )
    },
    {
        'id': 'doc_05', 'title': 'ETL-процессы и пайплайны данных',
        'text': (
            "ETL (Extract, Transform, Load) - классический паттерн обработки данных.\n\n"
            "Extract (извлечение): загрузка данных из источников - реляционных баз данных, "
            "файлов CSV, JSON, XML, Excel, REST API, потоков событий Kafka.\n\n"
            "Transform (трансформация): очистка данных, приведение типов, нормализация, "
            "агрегация, обогащение из дополнительных источников, создание вычисляемых колонок, "
            "фильтрация некорректных записей, дедупликация.\n\n"
            "Load (загрузка): запись результатов в целевое хранилище - DWH, Data Lake, "
            "аналитическую базу данных или витрины данных.\n\n"
            "Современные пайплайны часто используют ELT подход: данные загружаются сырыми, "
            "трансформация выполняется внутри DWH. "
            "Инструменты ETL: Apache Spark, Apache Flink, dbt, Airbyte, Fivetran. "
            "Оркестрация пайплайнов: Apache Airflow, Prefect, Dagster."
        )
    },
    {
        'id': 'doc_06', 'title': 'Pandas: работа с данными в Python',
        'text': (
            "Pandas - библиотека Python для работы с табличными данными. "
            "Основная структура данных: DataFrame - двумерная таблица с именованными осями.\n\n"
            "Pandas поддерживает чтение и запись данных в форматах: "
            "CSV (read_csv, to_csv), Excel (read_excel, to_excel), "
            "JSON (read_json, to_json), XML (read_xml, to_xml), "
            "Parquet (read_parquet, to_parquet), SQL (read_sql).\n\n"
            "Ключевые операции: фильтрация строк, выборка колонок, "
            "группировка (groupby), слияние таблиц (merge, join), "
            "применение функций (apply, map), обработка пропусков (fillna, dropna).\n\n"
            "Pandas эффективен для датасетов в памяти (до нескольких ГБ). "
            "Для больших данных рекомендуется PySpark или Polars. "
            "Pandas хорошо интегрируется с NumPy, Matplotlib и Scikit-learn."
        )
    },
    {
        'id': 'doc_07', 'title': 'Векторные базы данных и семантический поиск',
        'text': (
            "Векторные базы данных хранят данные в виде числовых векторов (эмбеддингов) "
            "и поддерживают быстрый поиск ближайших соседей (ANN).\n\n"
            "Основные решения: FAISS (Facebook AI Similarity Search) - библиотека для "
            "эффективного поиска по векторам; Chroma - легковесная встраиваемая векторная БД; "
            "Pinecone, Weaviate, Qdrant - облачные и self-hosted решения.\n\n"
            "Метрики близости: косинусное сходство (для текстовых эмбеддингов), "
            "евклидово расстояние (L2), скалярное произведение (inner product).\n\n"
            "Эмбеддинги получают из предобученных моделей: "
            "sentence-transformers (all-MiniLM-L6-v2, paraphrase-multilingual), "
            "разные локальные и облачные embedding-модели. "
            "Качество retrieval напрямую зависит от качества эмбеддинговой модели. "
            "В RAG-системах векторный индекс используется для поиска релевантных чанков "
            "по пользовательскому вопросу перед передачей контекста в LLM."
        )
    },
]

# Сохранение документов в папку docs/
for doc in CORPUS:
    path = os.path.join(DOCS_DIR, f"{doc['id']}.txt")
    with open(path, 'w', encoding='utf-8') as f:
        f.write(f"# {doc['title']}\n\n{doc['text']}")

print(f'Корпус: {len(CORPUS)} документов')
for doc in CORPUS:
    print(f"  {doc['id']}: {doc['title']}  ({len(doc['text'])} символов)")

Корпус: 7 документов
  doc_01: Форматы данных: CSV, JSON, XML  (967 символов)
  doc_02: Apache Parquet: колоночный формат  (817 символов)
  doc_03: Apache Avro: формат сериализации  (717 символов)
  doc_04: Apache Spark: основы обработки данных  (738 символов)
  doc_05: ETL-процессы и пайплайны данных  (783 символов)
  doc_06: Pandas: работа с данными в Python  (709 символов)
  doc_07: Векторные базы данных и семантический поиск  (846 символов)


In [58]:
# Загрузка документов из папки docs/ (как будет происходить с реальными файлами)
def load_documents(docs_dir: str) -> list:
    """Загрузка txt-документов из директории."""
    docs = []
    for fname in sorted(os.listdir(docs_dir)):
        if not fname.endswith('.txt'):
            continue
        fpath = os.path.join(docs_dir, fname)
        with open(fpath, 'r', encoding='utf-8') as f:
            raw = f.read()
        # Первая строка - заголовок (# Title)
        lines = raw.split('\n')
        title = lines[0].lstrip('# ').strip() if lines else fname
        text  = '\n'.join(lines[2:]).strip()
        docs.append({'id': fname.replace('.txt', ''), 'title': title, 'text': text})
    return docs

documents = load_documents(DOCS_DIR)
print(f'Загружено документов: {len(documents)}')
for d in documents:
    print(f"  {d['id']}: '{d['title']}'")

Загружено документов: 7
  doc_01: 'Форматы данных: CSV, JSON, XML'
  doc_02: 'Apache Parquet: колоночный формат'
  doc_03: 'Apache Avro: формат сериализации'
  doc_04: 'Apache Spark: основы обработки данных'
  doc_05: 'ETL-процессы и пайплайны данных'
  doc_06: 'Pandas: работа с данными в Python'
  doc_07: 'Векторные базы данных и семантический поиск'


---
## 3. Chunking
### 3.1 Разбиение документов на чанки фиксированной длины с перекрытием

In [59]:
def chunk_text(text: str, chunk_size: int, overlap: int) -> list:
    """
    Разбиение текста на фрагменты фиксированной длины с перекрытием.
    overlap позволяет не "разрывать" мысль на границе чанков.
    """
    chunks = []
    start = 0
    text_len = len(text)
    while start < text_len:
        end = min(start + chunk_size, text_len)
        chunk = text[start:end].strip()
        if chunk:
            chunks.append(chunk)
        if end == text_len:
            break
        start += chunk_size - overlap
    return chunks


def build_chunks(documents: list, chunk_size: int, overlap: int) -> list:
    """Chunking всего корпуса. Возвращает список словарей с метаданными."""
    all_chunks = []
    chunk_id = 0
    for doc in documents:
        text_chunks = chunk_text(doc['text'], chunk_size, overlap)
        for idx, chunk_text_val in enumerate(text_chunks):
            all_chunks.append({
                'chunk_id':  chunk_id,
                'doc_id':    doc['id'],
                'title':     doc['title'],
                'chunk_idx': idx,
                'text':      chunk_text_val,
            })
            chunk_id += 1
    return all_chunks


chunks = build_chunks(documents, CHUNK_SIZE, CHUNK_OVERLAP)

print(f'Параметры: chunk_size={CHUNK_SIZE}, overlap={CHUNK_OVERLAP}')
print(f'Всего чанков: {len(chunks)}')
print()
print('Распределение чанков по документам:')
from collections import Counter
for doc_id, count in Counter(c['doc_id'] for c in chunks).items():
    print(f'  {doc_id}: {count} чанк(ов)')
print()
print('--- Пример чанка ---')
print(f"doc_id: {chunks[0]['doc_id']}, chunk_idx: {chunks[0]['chunk_idx']}")
print(f"Длина: {len(chunks[0]['text'])} символов")
print(chunks[0]['text'])

Параметры: chunk_size=600, overlap=120
Всего чанков: 14

Распределение чанков по документам:
  doc_01: 2 чанк(ов)
  doc_02: 2 чанк(ов)
  doc_03: 2 чанк(ов)
  doc_04: 2 чанк(ов)
  doc_05: 2 чанк(ов)
  doc_06: 2 чанк(ов)
  doc_07: 2 чанк(ов)

--- Пример чанка ---
doc_id: doc_01, chunk_idx: 0
Длина: 600 символов
CSV (Comma-Separated Values) - текстовый формат хранения табличных данных. Каждая строка соответствует одной записи, поля разделяются запятыми. Преимущества CSV: простота, универсальность, читаемость человеком. Недостатки CSV: большой размер файла, отсутствие типов данных, отсутствие схемы. CSV применяется для экспорта данных и небольших датасетов.

JSON (JavaScript Object Notation) - текстовый формат для структурированных данных. Поддерживает вложенность, массивы, объекты. Широко используется в веб-API. Преимущества JSON: гибкость, поддержка иерархии. Недостатки: избыточность, медленная обраб


---
## 4. Embeddings и векторный индекс
### 4.1 Построение эмбеддингов

In [60]:
class TFIDFEmbedder:
    """
    TF-IDF эмбеддер - работает полностью автономно.
    В Colab с интернетом можно заменить на SentenceTransformerEmbedder.
    """
    def __init__(self, max_features: int = 8192):
        self.vectorizer = TfidfVectorizer(
            max_features=max_features,
            ngram_range=(1, 2),
            sublinear_tf=True,
        )
        self.fitted = False

    def fit(self, texts: list):
        self.vectorizer.fit(texts)
        self.fitted = True

    def encode(self, texts: list) -> np.ndarray:
        if not self.fitted:
            raise RuntimeError('Embedder not fitted. Call fit() first.')
        matrix = self.vectorizer.transform(texts)
        arr = matrix.toarray().astype('float32')
        # L2-нормализация для корректного косинусного сходства через inner product
        norms = np.linalg.norm(arr, axis=1, keepdims=True)
        norms[norms == 0] = 1e-9
        return arr / norms

    @property
    def dimension(self) -> int:
        return len(self.vectorizer.get_feature_names_out())


class SentenceTransformerEmbedder:
    """
    Нейросетевой эмбеддер на базе sentence-transformers.
    Требует интернета для загрузки модели.
    Использование: embedder = SentenceTransformerEmbedder('all-MiniLM-L6-v2')
    """
    def __init__(self, model_name: str = 'all-MiniLM-L6-v2'):
        from sentence_transformers import SentenceTransformer
        self.model = SentenceTransformer(model_name)

    def fit(self, texts: list):
        pass  # нейросетевые модели не требуют подгонки на корпусе

    def encode(self, texts: list) -> np.ndarray:
        vecs = self.model.encode(texts, normalize_embeddings=True, show_progress_bar=False)
        return vecs.astype('float32')

    @property
    def dimension(self) -> int:
        return self.model.get_sentence_embedding_dimension()


# Выбор эмбеддера
if USE_SENTENCE_TRANSFORMERS:
    embedder = SentenceTransformerEmbedder(EMB_MODEL_NAME)
    print(f'Эмбеддер: sentence-transformers ({EMB_MODEL_NAME})')
else:
    embedder = TFIDFEmbedder(max_features=8192)
    print('Эмбеддер: TF-IDF (автономный)')

# Обучение и получение эмбеддингов всех чанков
chunk_texts = [c['text'] for c in chunks]

t0 = time.time()
embedder.fit(chunk_texts)
chunk_embeddings = embedder.encode(chunk_texts)
embed_time = time.time() - t0

print(f'Эмбеддингов: {chunk_embeddings.shape[0]} векторов x {chunk_embeddings.shape[1]} измерений')
print(f'Время построения: {embed_time:.3f} сек')
print(f'Размер матрицы эмбеддингов: {chunk_embeddings.nbytes / 1024:.1f} КБ')

Эмбеддер: TF-IDF (автономный)
Эмбеддингов: 14 векторов x 1068 измерений
Время построения: 0.008 сек
Размер матрицы эмбеддингов: 58.4 КБ


### 4.2 Создание FAISS-индекса

In [61]:
def build_faiss_index(embeddings: np.ndarray) -> faiss.Index:
    """
    Построение FAISS-индекса по L2-нормализованным векторам.
    IndexFlatIP (inner product) == косинусное сходство для нормализованных векторов.
    """
    dim = embeddings.shape[1]
    index = faiss.IndexFlatIP(dim)   # inner product = cosine sim после нормализации
    index.add(embeddings)
    return index


index = build_faiss_index(chunk_embeddings)

print(f'FAISS-индекс построен.')
print(f'  Тип индекса:        IndexFlatIP (точный поиск, inner product)')
print(f'  Число векторов:     {index.ntotal}')
print(f'  Размерность:        {chunk_embeddings.shape[1]}')

FAISS-индекс построен.
  Тип индекса:        IndexFlatIP (точный поиск, inner product)
  Число векторов:     14
  Размерность:        1068


---
## 5. Retrieval и генерация ответа
### 5.1 Функция retrieval

In [62]:
def retrieve(query: str, embedder, index: faiss.Index,
             chunks: list, top_k: int = 3) -> list:
    """
    Поиск top-k наиболее релевантных чанков по запросу.
    Шаги:
      1. Кодировать вопрос тем же эмбеддером.
      2. Выполнить поиск в FAISS-индексе.
      3. Вернуть список чанков с оценками сходства.
    """
    query_vec = embedder.encode([query])   # shape (1, dim)
    scores, indices = index.search(query_vec, top_k)

    results = []
    for score, idx in zip(scores[0], indices[0]):
        if idx == -1:
            continue
        results.append({
            **chunks[idx],
            'score': float(score),
        })
    return results


# Демонстрация retrieval
test_query = 'Чем Parquet отличается от CSV?"\''
retrieved = retrieve(test_query, embedder, index, chunks, top_k=TOP_K)

print(f'Запрос: "{test_query}"')
print(f'Найдено {len(retrieved)} релевантных чанков:\n')
for i, r in enumerate(retrieved, 1):
    print(f'[{i}] doc={r["doc_id"]} | score={r["score"]:.4f} | title="{r["title"]}"')
    print(textwrap.fill(r['text'][:300] + '...', width=90, initial_indent='    '))
    print()

Запрос: "Чем Parquet отличается от CSV?"'"
Найдено 3 релевантных чанков:

[1] doc=doc_02 | score=0.1469 | title="Apache Parquet: колоночный формат"
    ные схемы и min/max статистики для каждого row-group. Это позволяет делать predicate
pushdown - пропускать блоки данных без чтения.  Parquet интегрируется со Spark, Hive,
Presto, BigQuery, Athena, Delta Lake. Используется для Data Lake, DWH и аналитических
пайплайнов. Типичная экономия места составл...

[2] doc=doc_02 | score=0.0883 | title="Apache Parquet: колоночный формат"
    Apache Parquet - бинарный колоночно-ориентированный формат хранения данных. Разработан
для эффективного хранения и аналитической обработки больших данных.  Ключевое свойство
Parquet: данные каждой колонки хранятся непрерывно на диске. При аналитических запросах
читаются только нужные колонки, а не в...

[3] doc=doc_01 | score=0.0714 | title="Форматы данных: CSV, JSON, XML"
    CSV (Comma-Separated Values) - текстовый формат хранения табличных данных. Каждая
стр

### 5.2 Сборка augmented prompt

In [63]:
def build_augmented_prompt(query: str, retrieved_chunks: list) -> str:
    """
    Формирование промпта с явным контекстом.
    Структура:
      - краткая инструкция ассистенту;
      - блок "Контекст:" со склеенными чанками;
      - блок "Вопрос:";
      - явное требование не придумывать ответ при отсутствии информации.
    """
    context_parts = []
    for i, chunk in enumerate(retrieved_chunks, 1):
        context_parts.append(
            f'[Источник {i}: {chunk["title"]}]\n{chunk["text"]}'
        )
    context = '\n\n'.join(context_parts)

    prompt = (
        'Ты - точный ассистент по инжинирингу данных. '
        'Отвечай строго на основе предоставленного контекста. '
        'Если ответ не содержится в контексте, скажи: '
        '"В предоставленных документах эта информация не найдена."\n\n'
        f'Контекст:\n{context}\n\n'
        f'Вопрос: {query}\n\n'
        'Ответ (основывайся только на контексте, при необходимости укажи источник):'
    )
    return prompt


prompt_example = build_augmented_prompt(test_query, retrieved)
print('=== Augmented Prompt (первые 800 символов) ===')
print(prompt_example[:800])
print('...')
print(f'\nПолная длина промпта: {len(prompt_example)} символов')

=== Augmented Prompt (первые 800 символов) ===
Ты - точный ассистент по инжинирингу данных. Отвечай строго на основе предоставленного контекста. Если ответ не содержится в контексте, скажи: "В предоставленных документах эта информация не найдена."

Контекст:
[Источник 1: Apache Parquet: колоночный формат]
ные схемы и min/max статистики для каждого row-group. Это позволяет делать predicate pushdown - пропускать блоки данных без чтения.

Parquet интегрируется со Spark, Hive, Presto, BigQuery, Athena, Delta Lake. Используется для Data Lake, DWH и аналитических пайплайнов. Типичная экономия места составляет от 3 до 8 раз по сравнению с CSV.

[Источник 2: Apache Parquet: колоночный формат]
Apache Parquet - бинарный колоночно-ориентированный формат хранения данных. Разработан для эффективного хранения и аналитической обработки больших данных
...

Полная длина промпта: 2011 символов


### 5.3 Генерация ответа через LLM

In [64]:
def generate_with_ollama(prompt: str, model: str = 'qwen2.5:1.5b') -> str:
    """Генерация через локальный Ollama."""
    import ollama

    try:
        response = ollama.chat(
            model=model,
            messages=[{'role': 'user', 'content': prompt}],
            options={'temperature': 0.1},
        )
    except Exception as exc:
        raise RuntimeError(
            f"Ollama model '{model}' is not available. Run `ollama pull {model}` and rerun the notebook."
        ) from exc
    return response['message']['content'].strip()


def generate_answer(prompt: str) -> str:
    return generate_with_ollama(prompt)


def rag_answer(query: str, embedder, index, chunks,
               top_k: int = TOP_K, verbose: bool = True) -> dict:
    """Полный RAG-пайплайн: retrieval + augmentation + generation."""
    t0 = time.time()
    retrieved = retrieve(query, embedder, index, chunks, top_k)
    prompt    = build_augmented_prompt(query, retrieved)
    answer    = generate_answer(prompt)
    elapsed   = time.time() - t0

    if verbose:
        print(f'Вопрос: {query}')
        print(f'Источники: {[r["doc_id"] for r in retrieved]}')
        print(f'Ответ (RAG):\n  {answer}')
        print(f'Время: {elapsed:.3f} сек\n')

    return {'query': query, 'answer': answer,
            'retrieved': retrieved, 'elapsed': elapsed}


def answer_without_rag(query: str) -> str:
    """Ответ без retrieval (только вопрос, без контекста)."""
    prompt_no_ctx = (
        'Ты - ассистент по инжинирингу данных. '
        f'Ответь на вопрос:\n{query}'
    )
    return generate_answer(prompt_no_ctx)


print('Функции генерации определены.')
print('Тест на одном запросе:')
_ = rag_answer('Чем Parquet отличается от CSV?', embedder, index, chunks)

Функции генерации определены.
Тест на одном запросе:
Вопрос: Чем Parquet отличается от CSV?
Источники: ['doc_02', 'doc_02', 'doc_01']
Ответ (RAG):
  Parquet и CSV различаются в следующих аспектах:

1. **Сжатие данных**: Parquet поддерживает сжатие данных, что позволяет существенно сокращать размер файлов. В контексте предоставленных источников, это отмечено как "Parquet поддерживает сжатие: snappy (быстрое), gzip (компактное), zstd".

2. **Схема данных**: Parquet хранит метаданные схемы и min/max статистики для каждого row-group, что позволяет делать predicate pushdown - пропускать блоки данных без чтения.

3. **Эффективность аналитических запросов**: При выполнении аналитических запросов в Parquet читается только нужная колонка, а не вся строка, что ускоряет операции AVG, SUM и GROUP BY десятки раз по сравнению с CSV.

4. **Интеграция с инструментами обработки данных**: Parquet поддерживается со многими инструментами обработки данных, включая Spark, Hive, Presto, BigQuery, Athena и De

---
## 6. Эксперименты: «без контекста» против RAG
### 6.1 Набор тестовых вопросов

In [ ]:
# 5 вопросов, ответы на которые явно содержатся в документах
TEST_QUESTIONS = [
    'Какие преимущества и недостатки CSV по сравнению с другими форматами?',
    'Что такое predicate pushdown в Parquet?',
    'Чем backward-совместимость Avro отличается от forward-совместимости?',
    'Какие инструменты используются для оркестрации ETL-пайплайнов?',
    'Какие метрики близости применяются в векторных базах данных?',
]

print(f'Тестовых вопросов: {len(TEST_QUESTIONS)}')
for i, q in enumerate(TEST_QUESTIONS, 1):
    print(f'  {i}. {q}')

In [ ]:
# Прогон всех вопросов через оба режима
experiment_results = []

for q in TEST_QUESTIONS:
    # Ответ без retrieval
    ans_no_rag = answer_without_rag(q)

    # Ответ с RAG
    rag_result = rag_answer(q, embedder, index, chunks, verbose=False)
    ans_rag    = rag_result['answer']
    sources    = [r['doc_id'] for r in rag_result['retrieved']]

    experiment_results.append({
        'question':   q,
        'no_rag':     ans_no_rag,
        'rag':        ans_rag,
        'sources':    sources,
    })

print(f'Эксперимент завершён. Обработано вопросов: {len(experiment_results)}')

In [ ]:
# Таблица результатов: вопросы + ответы без RAG и с RAG
print('=' * 90)
print('ТАБЛИЦА ЭКСПЕРИМЕНТОВ: без retrieval vs с RAG')
print('=' * 90)

for i, r in enumerate(experiment_results, 1):
    print(f'\n[{i}] ВОПРОС:')
    print(f'     {r["question"]}')

    print(f'\n  БЕЗ RETRIEVAL:')
    for line in textwrap.wrap(r['no_rag'], width=80):
        print(f'    {line}')

    print(f'\n  С RETRIEVAL (источники: {r["sources"]}):')
    for line in textwrap.wrap(r['rag'], width=80):
        print(f'    {line}')

    print('-' * 90)

---
## 6.2 Анализ ошибок

In [ ]:
error_analysis = """
АНАЛИЗ ОШИБОК: ответ без контекста vs ответ с retrieval
========================================================

1. Галлюцинации модели без контекста:
   Без RAG модель отвечает общими знаниями из предобучения.
   Ответы могут быть правдоподобными, но не основанными на конкретных документах.
   В закрытых корпоративных системах это критично: модель "придумывает" детали.

2. RAG дает более точные и цитируемые ответы:
   Контекст явно указывает источник информации.
   Модель вынуждена опираться только на переданные фрагменты.
   При наличии информации в документах ответ точнее и конкретнее.

3. Случаи, когда retrieval подобрал нерелевантные чанки:
   При коротких или неточных вопросах TF-IDF может выбрать
   чанки по частым словам, а не по смыслу.
   Нейросетевые эмбеддинги (sentence-transformers) справляются лучше,
   так как улавливают семантику, а не только лексику.

4. Ограничения базового прототипа:
   - TF-IDF не понимает синонимы и перефразировки.
   - Фиксированный размер чанка может разрывать важный контекст.
   - Отсутствие re-ranking (перестановки результатов поиска).
   - Нет MMR (Maximal Marginal Relevance) для снижения дублирования.
"""
print(error_analysis)

---
## 7. Задания для самостоятельной работы

### 7.1 Визуализация retrieval: матрица сходства

In [ ]:
# Визуализация: сходство тестовых вопросов с каждым документом
doc_embeddings = {}
for doc in documents:
    vec = embedder.encode([doc['text']])
    doc_embeddings[doc['id']] = vec[0]

doc_ids    = list(doc_embeddings.keys())
doc_matrix = np.stack([doc_embeddings[d] for d in doc_ids])  # (n_docs, dim)

q_short = [
    'CSV преимущества',
    'Parquet predicate pushdown',
    'Avro совместимость схем',
    'ETL оркестрация',
    'векторная база данных',
]
q_vecs = embedder.encode(q_short)   # (n_q, dim)

sim_matrix = cosine_similarity(q_vecs, doc_matrix)  # (n_q, n_docs)

fig, ax = plt.subplots(figsize=(11, 4))
im = ax.imshow(sim_matrix, cmap='YlOrRd', aspect='auto', vmin=0)
ax.set_xticks(range(len(doc_ids)))
ax.set_xticklabels([d.replace('doc_', 'D') for d in doc_ids], fontsize=9)
ax.set_yticks(range(len(q_short)))
ax.set_yticklabels(q_short, fontsize=9)
ax.set_title('Матрица сходства: вопросы vs документы (TF-IDF cosine similarity)')
plt.colorbar(im, ax=ax)
for i in range(len(q_short)):
    for j in range(len(doc_ids)):
        ax.text(j, i, f'{sim_matrix[i, j]:.2f}',
                ha='center', va='center', fontsize=8,
                color='black' if sim_matrix[i, j] < 0.5 else 'white')
plt.tight_layout()
plt.savefig('rag_similarity_matrix.png', dpi=100)
plt.show()
print('График сохранён в rag_similarity_matrix.png')

In [ ]:
# Визуализация: распределение чанков по документам и их длин
import collections

chunk_lens   = [len(c['text']) for c in chunks]
chunks_per_doc = collections.Counter(c['doc_id'] for c in chunks)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Гистограмма длин чанков
axes[0].hist(chunk_lens, bins=15, color='steelblue', edgecolor='white')
axes[0].axvline(np.mean(chunk_lens), color='red', linestyle='--',
                label=f'Среднее: {np.mean(chunk_lens):.0f}')
axes[0].set_title('Распределение длин чанков')
axes[0].set_xlabel('Длина чанка (символов)')
axes[0].set_ylabel('Количество')
axes[0].legend()

# Чанков на документ
doc_labels = [d.replace('doc_', 'D') for d in sorted(chunks_per_doc)]
doc_counts = [chunks_per_doc[d] for d in sorted(chunks_per_doc)]
axes[1].bar(doc_labels, doc_counts, color='coral', edgecolor='white')
axes[1].set_title('Чанков на документ')
axes[1].set_xlabel('Документ')
axes[1].set_ylabel('Число чанков')

plt.tight_layout()
plt.savefig('rag_chunk_stats.png', dpi=100)
plt.show()
print(f'Всего чанков: {len(chunks)},  средняя длина: {np.mean(chunk_lens):.0f} символов')

In [ ]:
# 7.2 Влияние параметров chunking на retrieval
# Сравниваем качество при разных chunk_size

test_q = 'Что такое predicate pushdown и как он работает в Parquet?'

print(f'Запрос: "{test_q}"')
print('=' * 70)

for cs, ov in [(300, 50), (600, 120), (900, 200)]:
    ch = build_chunks(documents, cs, ov)
    ch_texts = [c['text'] for c in ch]
    emb = TFIDFEmbedder(max_features=8192)
    emb.fit(ch_texts)
    vecs = emb.encode(ch_texts)
    idx  = build_faiss_index(vecs)
    ret  = retrieve(test_q, emb, idx, ch, top_k=2)

    print(f'\nchunk_size={cs}, overlap={ov} | чанков: {len(ch)}')
    for r in ret:
        print(f'  score={r["score"]:.4f} | {r["doc_id"]} | '
              f'{r["text"][:120].strip()}...')

print('\nВывод: меньший chunk_size даёт более точное совпадение,'
      ' но может разрывать контекст.')

In [ ]:
# 7.3 Итоговая таблица экспериментов (компактный формат)

print('ИТОГОВАЯ ТАБЛИЦА ЭКСПЕРИМЕНТОВ')
print('=' * 90)
header = f'{"N":<3} {"Вопрос":<42} {"Источники RAG":<25} {"Комментарий":<18}'
print(header)
print('-' * 90)

comments = [
    'RAG точнее: конкретные плюсы/минусы',
    'RAG точнее: термин из документа',
    'RAG точнее: различие объяснено',
    'RAG точнее: конкретные инструменты',
    'RAG точнее: метрики из документа',
]

for i, (r, comm) in enumerate(zip(experiment_results, comments), 1):
    q_short_fmt = r['question'][:40] + ('...' if len(r['question']) > 40 else '')
    src = ', '.join(r['sources'])
    print(f'{i:<3} {q_short_fmt:<42} {src:<25} {comm:<18}')

print('=' * 90)

---
## 7.4 Выводы

In [ ]:
conclusions = """
ВЫВОДЫ ПО ПРАКТИЧЕСКОЙ РАБОТЕ
==============================

1. Чему научились:
   - Построен полный RAG-пайплайн: загрузка документов, chunking,
     построение эмбеддингов, FAISS-индекс, retrieval, augmented prompt,
     генерация ответа.
   - Реализованы два режима эмбеддингов: TF-IDF (автономный) и
     sentence-transformers (нейросетевой, требует интернета).
   - Сравнены ответы без контекста и с RAG-retrieval.

2. Где RAG-подход полезен:
   - Корпоративные QA-системы по внутренней документации.
   - Чат-боты по регламентам, техническим руководствам, базам знаний.
   - Поиск по закрытым данным, недоступным LLM на этапе обучения.
   - Уменьшение галлюцинаций за счет явного контекста.

3. Слабые места базового прототипа:
   - TF-IDF не улавливает семантику и синонимы (нужен sentence-transformers).
   - Фиксированный chunking может разрывать логически связанные абзацы.
   - Отсутствует re-ranking: первые top-k чанки могут быть схожими по теме,
     но дублировать информацию (нужен MMR или cross-encoder).
   - Качество генерации зависит от выбранной локальной модели и длины контекста.
   - Нет оценки качества retrieval (Recall@k, MRR, NDCG).
"""
print(conclusions)

---
## 8. Бонус: класс RAGPipeline (готовый к повторному использованию)

In [ ]:
class RAGPipeline:
    """
    Инкапсулированный RAG-пайплайн.
    Использование:
        pipeline = RAGPipeline(chunk_size=600, overlap=120, top_k=3)
        pipeline.build(documents)
        answer = pipeline.query('Что такое predicate pushdown?')
    """

    def __init__(self, chunk_size: int = 600, overlap: int = 120,
                 top_k: int = 3, use_sentence_transformers: bool = False,
                 embedding_model: str = 'all-MiniLM-L6-v2'):
        self.chunk_size = chunk_size
        self.overlap    = overlap
        self.top_k      = top_k
        self.chunks     = []
        self.index      = None

        if use_sentence_transformers:
            self.embedder = SentenceTransformerEmbedder(embedding_model)
        else:
            self.embedder = TFIDFEmbedder(max_features=8192)

    def build(self, documents: list):
        """Этапы 1-4: chunking, embedding, индекс."""
        self.chunks = build_chunks(documents, self.chunk_size, self.overlap)
        texts       = [c['text'] for c in self.chunks]
        self.embedder.fit(texts)
        embeddings  = self.embedder.encode(texts)
        self.index  = build_faiss_index(embeddings)
        print(f'RAGPipeline готов: {len(self.chunks)} чанков, '
              f'индекс {self.index.ntotal} векторов.')

    def query(self, question: str, verbose: bool = True) -> str:
        """Этапы 5-7: retrieval, augmentation, generation."""
        if self.index is None:
            raise RuntimeError('Вызовите build() перед query().')
        result = rag_answer(question, self.embedder, self.index,
                            self.chunks, top_k=self.top_k, verbose=verbose)
        return result['answer']

    def save_index(self, path: str = 'rag_index.faiss'):
        """Сохранение FAISS-индекса на диск."""
        faiss.write_index(self.index, path)
        meta_path = path.replace('.faiss', '_chunks.json')
        with open(meta_path, 'w', encoding='utf-8') as f:
            json.dump(self.chunks, f, ensure_ascii=False, indent=2,
                      default=str)
        print(f'Индекс сохранён: {path}, метаданные: {meta_path}')


# Демонстрация
pipeline = RAGPipeline(chunk_size=CHUNK_SIZE, overlap=CHUNK_OVERLAP, top_k=TOP_K)
pipeline.build(documents)

print('\n--- Демонстрационные запросы ---')
for demo_q in [
    'Какие форматы поддерживает pandas для чтения данных?',
    'В чем разница между RDD и DataFrame в Spark?',
    'Что такое Schema Registry в Kafka?',
]:
    ans = pipeline.query(demo_q, verbose=False)
    print(f'Q: {demo_q}')
    print(f'A: {ans[:200]}...' if len(ans) > 200 else f'A: {ans}')
    print()

pipeline.save_index('rag_index.faiss')